# TradeSense · Component 03 · Layer 1 rebuild from raw trades (v1)

**Purpose.** (1) Look in your Drive for raw trade files. (2) If none are there, download the public Binance spot `aggTrades` day files (data.binance.vision) for BTCUSDT and ETHUSDT. (3) Rebuild the six Layer 1 features with the frozen contract formulas. (4) Compare the rebuild with the existing Layer 1 on 2024-05-01, 06-01 and 07-01 (this settles the open raw-tick reconciliation). (5) Build the same bars for 2024-08-01 .. 2024-10-31 for the later models and the single test opening.

**Safe.** Existing Layer 1 files are only read. Output goes to `layer1_rebuild_v1/`. Each day is processed once and saved, so *Run all* again continues where it stopped (zip files are deleted after processing to save disk). CPU runtime is enough. A day is about 1 to 3 minutes. `MAX_DAYS_THIS_RUN` is 100, so one Run all builds everything (about 20 minutes).

In [ ]:
import os, io, re, json, time, glob, zipfile, hashlib, warnings, urllib.request, urllib.error
from pathlib import Path
from datetime import date, timedelta
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab -> using TRADESENSE_BASE.')
BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets')); assert BASE.is_dir(), BASE
OUT = BASE / 'layer1_rebuild_v1'; (OUT/'bars').mkdir(parents=True, exist_ok=True); (OUT/'zips').mkdir(exist_ok=True)
SYMBOLS = ['BTCUSDT', 'ETHUSDT']
FLOOR = {'BTCUSDT': 2.5399, 'ETHUSDT': 27.7658}       # frozen volume_acceleration floors
LARGE = {'BTCUSDT': 0.1, 'ETHUSDT': 2.0}               # frozen large-trade thresholds (base units)
RECONCILE_DAYS = [date(2024,5,1), date(2024,6,1), date(2024,7,1)]
NEW_START, NEW_END = date(2024,8,1), date(2024,10,31)
MAX_DAYS_THIS_RUN = int(os.environ.get('MAX_DAYS', 100))
LINES = []
def say(s=''): print(s); LINES.append(str(s))

Mounted at /content/drive


## 1 · Look for raw trades in Drive

In [ ]:
# 1. look for raw trade files already in Drive (depth <= 4, names with trade/tardis)
pat = re.compile(r'(aggtrade|trades|tardis)', re.I); ext = ('.csv','.gz','.zip','.parquet','.csv.gz')
found = []
roots = [BASE, BASE.parent]
for r in roots:
    for dp, dn, fn in os.walk(r):
        if str(Path(dp)).count(os.sep) - str(r).count(os.sep) > 4: dn[:] = []; continue
        for f in fn:
            if pat.search(f) and f.lower().endswith(ext):
                p = Path(dp)/f; found.append((str(p), p.stat().st_size/1e6))
seen=set(); found=[x for x in found if not (x[0] in seen or seen.add(x[0]))]
say(f'raw trade-like files found in Drive: {len(found)}')
for p,mb in found[:40]: say(f'  {mb:9.1f} MB  {p}')
# existing Layer 1 matrices (read only)
L1 = {}
for s in SYMBOLS:
    c = sorted(glob.glob(str(BASE/'**'/f'{s.lower()}_l1_feature_matrix_90d.parquet'), recursive=True))
    c = [x for x in c if 'backup' not in x.lower()]
    L1[s] = c[0] if c else None
    say(f'Layer 1 matrix {s}: {L1[s]}')

raw trade-like files found in Drive: 13
       27.9 MB  /content/drive/MyDrive/research datasets/TGNN_Tardis_Binance_Spot_BTCUSDT_2024-05-01_v1/raw/binance_trades_2024-05-01_BTCUSDT.csv.gz
       15.9 MB  /content/drive/MyDrive/research datasets/TGNN_Tardis_Binance_Spot_ETHUSDT_2024-05-01_v1/raw/binance_trades_2024-05-01_ETHUSDT.csv.gz
        7.0 MB  /content/drive/MyDrive/research datasets/TGNN_Tardis_Binance_Spot_BTCUSDT_2024-06-01_v1/raw/binance_trades_2024-06-01_BTCUSDT.csv.gz
        4.7 MB  /content/drive/MyDrive/research datasets/TGNN_Tardis_Binance_Spot_ETHUSDT_2024-06-01_v1/raw/binance_trades_2024-06-01_ETHUSDT.csv.gz
       14.1 MB  /content/drive/MyDrive/research datasets/TGNN_Tardis_Binance_Spot_BTCUSDT_2024-07-01_v1/raw/binance_trades_2024-07-01_BTCUSDT.csv.gz
        8.1 MB  /content/drive/MyDrive/research datasets/TGNN_Tardis_Binance_Spot_ETHUSDT_2024-07-01_v1/raw/binance_trades_2024-07-01_ETHUSDT.csv.gz
       24.3 MB  /content/drive/MyDrive/research datasets/TGNN_Tard

## 2 · Feature contract and unit tests

In [ ]:
# 2. the frozen feature contract, applied to one continuous run of 1-minute bars
def bars_from_trades(ts_ms, price, qty, buyer_maker, first_id=None, last_id=None, large=0.1):
    """1-minute bars from (agg)trades. taker BUY volume = trades with is_buyer_maker == False."""
    ts_ms = np.asarray(ts_ms, dtype=np.int64)
    ts_ms = np.where(ts_ms > 10**14, ts_ms // 1000, ts_ms)           # microseconds -> ms (newer files)
    minute = ts_ms // 60000
    df = pd.DataFrame({'m': minute, 'p': price, 'q': qty, 'bm': buyer_maker})
    df['buy'] = np.where(~df.bm, df.q, 0.0); df['sell'] = np.where(df.bm, df.q, 0.0)
    df['big'] = (df.q >= large).astype(np.int64)
    if first_id is not None: df['ids'] = np.asarray(last_id) - np.asarray(first_id) + 1
    g = df.groupby('m', sort=True)
    b = pd.DataFrame({'open': g.p.first(), 'high': g.p.max(), 'low': g.p.min(), 'close': g.p.last(),
                      'volume': g.q.sum(), 'buy_volume': g.buy.sum(), 'sell_volume': g.sell.sum(),
                      'agg_count': g.q.size(), 'large_trade_count': g.big.sum()})
    b['trade_id_count'] = g.ids.sum() if first_id is not None else np.nan
    full = np.arange(b.index.min(), b.index.max()+1)
    b = b.reindex(full)
    b['close'] = b.close.ffill();
    for c in ['open','high','low']: b[c] = b[c].fillna(b.close)
    for c in ['volume','buy_volume','sell_volume','agg_count','large_trade_count','trade_id_count']: b[c] = b[c].fillna(0)
    b.index = pd.to_datetime(b.index*60000, unit='ms', utc=True); b.index.name = 'datetime'
    return b

def contract_features(b, symbol, trades_col='agg_count'):
    """Six Layer 1 features. Bars 0..14 of a run: volume_acceleration = 0.0 ; log_returns bar0 = 0."""
    o = pd.DataFrame(index=b.index)
    c = b.close.to_numpy(); v = b.volume.to_numpy()
    lr = np.zeros(len(b)); lr[1:] = np.log(c[1:]/c[:-1]); o['log_returns'] = lr
    o['trade_flow_imbalance'] = (b.buy_volume-b.sell_volume)/(b.buy_volume+b.sell_volume+1e-8)
    o['hl_spread_ratio'] = (b.high-b.low)/b.close
    o['rush_order_density'] = b[trades_col]/(b.volume+1e-8)
    va = np.zeros(len(b)); s = pd.Series(v)
    mu = s.shift(1).rolling(15).mean().to_numpy(); sd = s.shift(1).rolling(15).std(ddof=1).to_numpy()
    ok = np.arange(len(b)) >= 15
    va[ok] = (v[ok]-mu[ok])/np.maximum(sd[ok], FLOOR[symbol]); o['volume_acceleration'] = va
    o['large_trade_count'] = b.large_trade_count
    return o

def features_by_runs(b, symbol, trades_col='agg_count'):
    """Split at gaps in the day list (a missing day starts a new run -> warm-up masked again)."""
    gap = b.index.to_series().diff() != pd.Timedelta('1min'); rid = gap.cumsum().to_numpy()
    parts = [contract_features(b[rid==k], symbol, trades_col) for k in np.unique(rid)]
    return pd.concat(parts)

# unit tests on synthetic trades
rng = np.random.default_rng(0); n = 6000
t0 = 1714521600000; ts = np.sort(t0 + rng.integers(0, 40*60000, n))
px = 60000 + np.cumsum(rng.normal(0, 2, n)); q = rng.exponential(0.05, n); bm = rng.random(n) < 0.5
b = bars_from_trades(ts, px, q, bm, np.arange(n), np.arange(n)+rng.integers(0,3,n), large=0.1)
assert len(b) == 40 and abs(b.volume.sum()-q.sum()) < 1e-9 and abs(b.buy_volume.sum()+b.sell_volume.sum()-q.sum()) < 1e-9
f = contract_features(b, 'BTCUSDT'); assert (f.volume_acceleration.iloc[:15] == 0).all() and f.log_returns.iloc[0] == 0
i = 20; w = b.volume.iloc[i-15:i].to_numpy(); ref = (b.volume.iloc[i]-w.mean())/max(w.std(ddof=1), FLOOR['BTCUSDT'])
assert abs(f.volume_acceleration.iloc[i]-ref) < 1e-12
assert f.trade_flow_imbalance.between(-1,1).all() and (f.large_trade_count >= 0).all()
print('unit tests passed (bars, buy/sell split, vol_acc contract, warm-up mask)')

unit tests passed (bars, buy/sell split, vol_acc contract, warm-up mask)


## 3 · Download and rebuild day by day

In [ ]:
# 3. one day = (Tardis file already in Drive, else download aggTrades zip, or use a zip you placed in zips/) -> bars -> save -> delete zip. Resumable.
URL = 'https://data.binance.vision/data/spot/daily/aggTrades/{s}/{s}-aggTrades-{d}.zip'
def day_path(s, d): return OUT/'bars'/f'{s}_{d.isoformat()}_bars.parquet'
def process_day(s, d):
    p = day_path(s, d)
    if p.exists(): return 'cached'
    tf = [x for x, _ in found if s in Path(x).name and d.isoformat() in Path(x).name and 'trades' in Path(x).name.lower() and 'orderbook' not in x.lower()]
    tf = sorted(tf, key=lambda x: (str(BASE) not in x, len(x)))
    df = None
    for cand in tf:   # Tardis raw trades already in Drive: columns timestamp(us), side(taker), price, amount, id
        try: df = pd.read_csv(cand, usecols=['timestamp','side','price','amount','id']); break
        except Exception: df = None
    if df is not None:
        b = bars_from_trades(df.timestamp.to_numpy(), df.price.to_numpy(), df.amount.to_numpy(), (df.side=='sell').to_numpy(),
                             df['id'].to_numpy(), df['id'].to_numpy(), large=LARGE[s])
        t0 = pd.Timestamp(d, tz='UTC'); b = b[(b.index >= t0) & (b.index < t0 + pd.Timedelta('1D'))]
        b.to_parquet(p); return f'ok (Tardis file in Drive) {len(b)} bars, {len(df):,} trades'
    zp = OUT/'zips'/f'{s}-aggTrades-{d.isoformat()}.zip'
    if not zp.exists():
        try: urllib.request.urlretrieve(URL.format(s=s, d=d.isoformat()), zp)
        except Exception as e: return f'DOWNLOAD FAILED: {type(e).__name__} {e}'
    with zipfile.ZipFile(zp) as z, z.open(z.namelist()[0]) as fh:
        df = pd.read_csv(fh, header=None, usecols=[1,2,3,4,5,6])
    df.columns = ['price','qty','first','last','ts','bm']
    b = bars_from_trades(df.ts.to_numpy(), df.price.to_numpy(), df.qty.to_numpy(), df.bm.astype(bool).to_numpy(),
                         df['first'].to_numpy(), df['last'].to_numpy(), large=LARGE[s])
    t0 = pd.Timestamp(d, tz='UTC'); b = b[(b.index >= t0) & (b.index < t0 + pd.Timedelta('1D'))]
    b.to_parquet(p); zp.unlink(missing_ok=True)
    return f'ok {len(b)} bars, {len(df):,} aggTrades'

try:
    r = urllib.request.urlopen(urllib.request.Request(URL.format(s='BTCUSDT', d='2024-05-01'), method='HEAD'), timeout=30)
    say(f'data.binance.vision reachable (HTTP {r.status}, {int(r.headers.get("Content-Length",0))/1e6:.0f} MB for BTC 2024-05-01)'); REACH = True
except Exception as e:
    REACH = False
    say(f'data.binance.vision NOT reachable from this runtime: {type(e).__name__} {e}')
    say('FALLBACK: download the day files in your own browser (e.g. https://data.binance.vision/data/spot/daily/aggTrades/BTCUSDT/BTCUSDT-aggTrades-2024-05-01.zip) '
        'and put the .zip files, with their original names, into Drive folder  layer1_rebuild_v1/zips/ . Run this notebook again: they are processed automatically.')

todo = []
for d in RECONCILE_DAYS: todo += ([d - timedelta(days=1), d] if d != date(2024,5,1) else [d])
d = NEW_START
while d <= NEW_END: todo.append(d); d += timedelta(days=1)
todo = list(dict.fromkeys(todo)); jobs = [(s, d) for d in todo for s in SYMBOLS]
done = 0
for s, d in jobs:
    if day_path(s, d).exists(): continue
    if done >= MAX_DAYS_THIS_RUN*len(SYMBOLS): break
    zp = OUT/'zips'/f'{s}-aggTrades-{d.isoformat()}.zip'
    has_tardis = any(s in Path(x).name and d.isoformat() in Path(x).name and 'trades' in Path(x).name.lower() and 'orderbook' not in x.lower() for x,_ in found)
    if not REACH and not zp.exists() and not has_tardis: continue
    t = time.time(); res = process_day(s, d); say(f'{s} {d}: {res} ({time.time()-t:.0f}s)'); done += 1
have = sorted(p.name for p in (OUT/'bars').glob('*.parquet'))
say(f'day files now available: {len(have)} of {len(jobs)} planned')

data.binance.vision reachable (HTTP 200, 26 MB for BTC 2024-05-01)
BTCUSDT 2024-08-12: ok 1440 bars, 1,588,406 aggTrades (5s)
ETHUSDT 2024-08-12: ok 1440 bars, 1,353,175 aggTrades (3s)
BTCUSDT 2024-08-13: ok 1440 bars, 1,146,007 aggTrades (3s)
ETHUSDT 2024-08-13: ok 1440 bars, 1,036,153 aggTrades (4s)
BTCUSDT 2024-08-14: ok 1440 bars, 1,178,962 aggTrades (4s)
ETHUSDT 2024-08-14: ok 1440 bars, 1,002,207 aggTrades (4s)
BTCUSDT 2024-08-15: ok 1440 bars, 1,540,600 aggTrades (4s)
ETHUSDT 2024-08-15: ok 1440 bars, 992,329 aggTrades (4s)
BTCUSDT 2024-08-16: ok 1440 bars, 1,159,666 aggTrades (3s)
ETHUSDT 2024-08-16: ok 1440 bars, 865,070 aggTrades (3s)
BTCUSDT 2024-08-17: ok 1440 bars, 483,972 aggTrades (1s)
ETHUSDT 2024-08-17: ok 1440 bars, 303,046 aggTrades (1s)
BTCUSDT 2024-08-18: ok 1440 bars, 663,828 aggTrades (3s)
ETHUSDT 2024-08-18: ok 1440 bars, 480,173 aggTrades (2s)
BTCUSDT 2024-08-19: ok 1440 bars, 977,050 aggTrades (3s)
ETHUSDT 2024-08-19: ok 1440 bars, 656,218 aggTrades (2s)
BTCUS

## 4 · Reconcile with the existing Layer 1

In [ ]:
# 4. reconcile the rebuild with the existing Layer 1 on the three reference days
COLS = ['log_returns','trade_flow_imbalance','hl_spread_ratio','rush_order_density','volume_acceleration','large_trade_count']
def load_bars(s):
    ps = sorted((OUT/'bars').glob(f'{s}_*_bars.parquet'))
    return pd.concat([pd.read_parquet(p) for p in ps]).sort_index() if ps else None
REC = {}
for s in SYMBOLS:
    b = load_bars(s)
    if b is None or L1.get(s) is None: say(f'{s}: nothing to reconcile yet (bars: {b is not None}, Layer 1 matrix: {L1.get(s)})'); continue
    ex = pd.read_parquet(L1[s])
    if 'datetime' in ex.columns: ex = ex.set_index('datetime')
    ex.index = pd.to_datetime(ex.index, utc=True)
    for tc in ['agg_count', 'trade_id_count']:
        f = features_by_runs(b, s, tc)
        for d in RECONCILE_DAYS:
            sl = slice(pd.Timestamp(d, tz='UTC'), pd.Timestamp(d, tz='UTC')+pd.Timedelta('23h59m'))
            a, e = f.loc[sl], ex.loc[sl, COLS]
            if len(a) < 1440 or len(e) < 1440: continue
            warm = d != date(2024,5,1)          # first Layer 1 day: our run also starts there
            row = {}
            for c in COLS:
                x, y = a[c].to_numpy(), e[c].to_numpy()
                if c == 'volume_acceleration': x, y = x[15:], y[15:]
                row[c] = (float(np.max(np.abs(x-y))), float(np.mean(np.isclose(x, y, rtol=1e-6, atol=1e-9))))
            REC[(s, tc, str(d))] = row
            say(f'{s} {d} trades={tc}: ' + '  '.join(f'{c.split("_")[0]}:{row[c][1]*100:5.1f}%' for c in COLS))
say('')
say('Reading the table: 100% = rebuilt feature equals the stored Layer 1 feature (rel tol 1e-6). A feature that is below 100% for BOTH trade counts '
    'means the stored feature was built by a different rule (for example full trades instead of aggTrades); the exact differences are stored in the report.')

BTCUSDT 2024-05-01 trades=agg_count: log: 99.9%  trade: 99.9%  hl:100.0%  rush: 99.9%  volume: 99.9%  large:100.0%
BTCUSDT 2024-06-01 trades=agg_count: log:100.0%  trade:100.0%  hl:100.0%  rush:100.0%  volume: 76.9%  large:100.0%
BTCUSDT 2024-07-01 trades=agg_count: log: 99.9%  trade: 99.9%  hl:100.0%  rush: 99.9%  volume: 97.3%  large: 99.9%
BTCUSDT 2024-05-01 trades=trade_id_count: log: 99.9%  trade: 99.9%  hl:100.0%  rush: 99.9%  volume: 99.9%  large:100.0%
BTCUSDT 2024-06-01 trades=trade_id_count: log:100.0%  trade:100.0%  hl:100.0%  rush:100.0%  volume: 76.9%  large:100.0%
BTCUSDT 2024-07-01 trades=trade_id_count: log: 99.9%  trade: 99.9%  hl:100.0%  rush: 99.9%  volume: 97.3%  large: 99.9%
ETHUSDT 2024-05-01 trades=agg_count: log: 99.9%  trade: 99.9%  hl:100.0%  rush: 99.9%  volume: 99.9%  large:100.0%
ETHUSDT 2024-06-01 trades=agg_count: log:100.0%  trade: 99.9%  hl:100.0%  rush: 99.9%  volume: 99.9%  large:100.0%
ETHUSDT 2024-07-01 trades=agg_count: log:100.0%  trade:100.0%  hl

## 5 · Report and share block

In [ ]:
# 5. report + share block
rep = {'created': pd.Timestamp.now(tz='UTC').isoformat(), 'reachable_binance_public': bool(REACH),
       'day_files': len(list((OUT/'bars').glob('*.parquet'))), 'raw_files_in_drive': len(found),
       'reconcile': {f'{k[0]}|{k[1]}|{k[2]}': v for k, v in REC.items()}}
(OUT/'rebuild_report.json').write_text(json.dumps(rep, indent=1))
block = '\n'.join(LINES); (OUT/'rebuild_share_block.txt').write_text(block)
print('\n===== SHARE BLOCK (copy everything below, or attach layer1_rebuild_v1/rebuild_share_block.txt) =====')
print(block[-6000:])


===== SHARE BLOCK (copy everything below, or attach layer1_rebuild_v1/rebuild_share_block.txt) =====
ggTrades (2s)
BTCUSDT 2024-09-26: ok 1440 bars, 1,747,533 aggTrades (4s)
ETHUSDT 2024-09-26: ok 1440 bars, 1,258,692 aggTrades (5s)
BTCUSDT 2024-09-27: ok 1440 bars, 1,482,549 aggTrades (4s)
ETHUSDT 2024-09-27: ok 1440 bars, 1,078,220 aggTrades (4s)
BTCUSDT 2024-09-28: ok 1440 bars, 659,443 aggTrades (2s)
ETHUSDT 2024-09-28: ok 1440 bars, 504,910 aggTrades (3s)
BTCUSDT 2024-09-29: ok 1440 bars, 650,124 aggTrades (3s)
ETHUSDT 2024-09-29: ok 1440 bars, 620,098 aggTrades (2s)
BTCUSDT 2024-09-30: ok 1440 bars, 1,548,282 aggTrades (4s)
ETHUSDT 2024-09-30: ok 1440 bars, 1,117,879 aggTrades (3s)
BTCUSDT 2024-10-01: ok (Tardis file in Drive) 1440 bars, 5,607,068 trades (12s)
ETHUSDT 2024-10-01: ok (Tardis file in Drive) 1440 bars, 4,525,441 trades (10s)
BTCUSDT 2024-10-02: ok 1440 bars, 1,818,421 aggTrades (4s)
ETHUSDT 2024-10-02: ok 1440 bars, 1,915,939 aggTrades (5s)
BTCUSDT 2024-10-03: ok 1